In [ ]:
# ---
# This notebook performs the following tasks:
# - Loads multiple CSV files containing SPARQL queries.
# - Filters out non-SELECT queries (ASK, CONSTRUCT, DESCRIBE).
# - Removes duplicate SELECT queries (case and whitespace insensitive).
# - Saves unique SELECT queries to a new CSV file.
# - Prints summary statistics about the filtering process.
# ---

import pandas as pd
import glob
import re


In [ ]:
def is_select_query(query: str) -> bool:
    """
    Returns True if the query is a SELECT query.
    Filters out ASK, CONSTRUCT, and DESCRIBE queries.
    """
    normalized = re.sub(r"\s+", " ", query).strip().lower()
    # Exclude queries containing ASK, CONSTRUCT, or DESCRIBE
    if any(kw in normalized for kw in ["ask", "construct", "describe"]):
        return False
    return True

def process_csv_files(csv_files):
    """
    Processes CSV files to extract unique SELECT queries.
    Returns a DataFrame and statistics.
    """
    all_queries = set()
    combined_df = pd.DataFrame(columns=["text"])
    stats = {
        "total": 0,
        "non_select": 0,
        "duplicate": 0
    }

In [ ]:
def process_csv_files(csv_files):
    """
    Processes CSV files to extract unique SELECT queries.
    Returns a DataFrame and statistics.
    """
    all_queries = set()
    combined_df = pd.DataFrame(columns=["text"])
    stats = {
        "total": 0,
        "non_select": 0,
        "duplicate": 0
    }

    for file in csv_files:
        df = pd.read_csv(file)
        if "text" not in df.columns:
            continue

        for query in df["text"].dropna().astype(str):
            stats["total"] += 1
            normalized_query = re.sub(r"\s+", " ", query).strip().lower()

            if not is_select_query(query):
                stats["non_select"] += 1
                continue

            if normalized_query in all_queries:
                stats["duplicate"] += 1
                continue

            all_queries.add(normalized_query)
            combined_df = pd.concat([combined_df, pd.DataFrame({"text": [query]})], ignore_index=True)

    return combined_df, stats


In [ ]:
# Main execution
csv_files = glob.glob("lsq_data/*.csv")
combined_df, stats = process_csv_files(csv_files)
combined_df.to_csv("combined_unique_select_queries.csv", index=False)

# Print summary statistics
print("=== Summary ===")
print(f"Total queries processed: {stats['total']}")
print(f"Non-SELECT queries skipped: {stats['non_select']}")
print(f"Duplicate SELECT queries skipped: {stats['duplicate']}")
print(f"Unique SELECT queries saved: {len(combined_df)}")